# **LangChain으로 프롬프트 생성하기(Prompt)**

## 실습 소개

LangChain은 다양한 구성 요소를 유연하게 연결하여, 거대 언어 모델(LLM)을 더 정밀하게 활용할 수 있게 도와주는 도구로 다양한 모듈(PromptTemplate, Document Loader, Parser 등)을 제공합니다.

다양한 LLM의 API를 직접 호출하는 방식은 단순하지만, 프롬프트를 동적으로 만들거나 반복적으로 활용하기엔 비효율적입니다. LangChain을 활용하면 **입력 변수 기반의 프롬프트 생성**, **모델 응답의 일관된 처리**, **반복적인 작업의 효율적 구성**이 가능해집니다.

이 실습에서는 LangChain을 활용하여 **효과적인 프롬프트를 설계**하고, 이를 **LLM에 적용해 자연스러운 응답을 생성하는 방법**을 배웁니다. 

## 학습 목표
- LangChain에서 제공하는 `PromptTemplate`의 구조와 사용법 이해
- LLM에 입력할 프롬프트를 구조화하고 활용하는 방법 학습
- 프롬프트 설계 → 모델 응답 생성 → 결과 확인의 흐름 경험

---

## 프롬프트 생성하기 
LangChain은 LLM에게 효과적으로 질문할 수 있도록 프롬프트 템플릿 기능을 제공하고 있습니다. 사용자가 원하는 작업에 맞춘 프롬프트를 쉽게 설정할 수 있어, 반복되는 작업을 간소화하고 일관성 있는 답변을 생성할 수 있습니다. 

### PromptTemplate
**PromptTemplate**은 일반적인 언어 모델에 사용하는 프롬프트 템플릿입니다. 특정 작업에 필요한 단일 문자열 형식의 프롬프트를 작성할 때 사용합니다. 간단한 질의응답, 문서 요약, 데이터 분석 등 다양한 작업에서 사용할 수 있습니다. 

In [1]:
from langchain_core.prompts import PromptTemplate

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


간단한 사용법은 다음과 같습니다. 하지만 이렇게 사용하기 보다는 일반적인 문자열을 이용하는 방법이 좋아보입니다. 

In [2]:
prompt = PromptTemplate.from_template("카메라를 홍보하기 위한 추천 문구를 작성해 줘")
prompt.format()

'카메라를 홍보하기 위한 추천 문구를 작성해 줘'

포맷(format) 기능을 이용하면 원하는 키워드에 단어를 매칭할 수 있습니다. 

파이썬의 기본 문자열에서도 지원하는 기능이지만 템플릿을 사용하면 코드 가독성, 재사용성 등 LangChain을 이용한 애플리케이션을 개발할 때 더 용이해지는 장점이 있습니다. 

In [3]:
template = "{product}를(을) 홍보하기 위한 추천 문구를 작성해줘"

prompt = PromptTemplate.from_template(template)
prompt.format(product="카메라")

'카메라를(을) 홍보하기 위한 추천 문구를 작성해줘'

다른 사용 방법은 다음과 같습니다. 

In [4]:
template = "{product}를(을) {task}하기 위한 추천 문구를 작성해 줘"

prompt = PromptTemplate(
    input_variables = ["product", "task"],
    template = template
)

prompt.format(product="노트북", task="충전")

'노트북를(을) 충전하기 위한 추천 문구를 작성해 줘'

다음은 실제 크롤링으로 수집한 영화 리뷰 데이터 중 일부입니다. PromptTemplate을 이용하여 감정 분석을 위한 프롬프트 템플릿을 작성해 보세요.

In [6]:
review_list = ["너무 난해하고 복잡한 영화였어요", "너무 재미있었어요!", "지루해요"]

template = "문구 : {text} \n문구를 보고 {result1} 또는 {result2} 중에 하나로 분류해 줘"

prompt = PromptTemplate(
    input_variables = {"text", "result1", "result2"},
    template = template
)

prompt.format(text=review_list[0], result1="긍정", result2="부정")

'문구 : 너무 난해하고 복잡한 영화였어요 \n문구를 보고 긍정 또는 부정 중에 하나로 분류해 줘'

In [7]:
for review_txt in review_list :
    
    prompt_txt = prompt.format(
        text = review_txt,
        result1 = "긍정",
        result2 = "부정"
    )

    print(prompt_txt)

문구 : 너무 난해하고 복잡한 영화였어요 
문구를 보고 긍정 또는 부정 중에 하나로 분류해 줘
문구 : 너무 재미있었어요! 
문구를 보고 긍정 또는 부정 중에 하나로 분류해 줘
문구 : 지루해요 
문구를 보고 긍정 또는 부정 중에 하나로 분류해 줘


PromptTemplate을 이용하여 자신만의 프롬프트를 만들어 보세요.

In [8]:
my_prompt = PromptTemplate(
    input_variables = {"text", "result1", "result2"},
    template = template
)

### ChatPromptTemplate
대화형 언어 모델과 상호작용을 위해 설계된 템플릿입니다. 프롬프트를 여러 개의 메세지 형식으로 나눌 수 있습니다. 쉽게 생각하면 모델에게 역할(e.g. 당신은 법률 전문가입니다 or 당신은 제조 분야의 공장장입니다)을 부여할 수 있으며 복잡한 작업을 여러 메세지로 나누어 단계별로 모델에게 안내할 수 있는 프롬프트 템플릿입니다. 

In [9]:
from langchain_core.prompts import ChatPromptTemplate

ChatPromptTemplate는 하나의 프롬프트에 여려 메세지를 포함하여 구성할 수 있습니다. 각 메세지에는 메세지의 역할을 지정하여 대화의 흐름과 맥락을 반영할 수 있습니다. 일반적으로 세 개의 유형이 있습니다. 

- system (SystemMessage) : 시스템 메세지, 모델의 행동이나 역할을 지정하는 최초 설정에 대한 메세지입니다. 
- human (HumanMessage) : 사용자 메세지, 모델에 질문하거나 지시하는 역할에 대한 메세지입니다. 
- ai (AIMessage) : AI (LLM) 모델 메세지, 사용자 메세지(프롬프트)에 대해 모델이 답변한 메시지입니다.

In [10]:
chat_prompt_template = ChatPromptTemplate(
    [
        ("system", "당신은 한국 역사에 대해 전문적인 지식을 가진 전문 AI 어시스턴트입니다. 이름은 {name}입니다."),
        ("human", "안녕하세요 저는 {history_keyword}에 대해 궁금해요!"),
        ("ai", "같이 한 번 알아볼까요? 어떤게 궁금하신가요?"),
        ("human", "{question}")
    ]
)

In [12]:
chat_prompt = chat_prompt_template.format_messages(
    name = "헬로빗",
    history_keyword = "조선",
    question = "세종대왕 님의 음식 취향이 궁금해요"
)

chat_prompt

[SystemMessage(content='당신은 한국 역사에 대해 전문적인 지식을 가진 전문 AI 어시스턴트입니다. 이름은 헬로빗입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='안녕하세요 저는 조선에 대해 궁금해요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='같이 한 번 알아볼까요? 어떤게 궁금하신가요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세종대왕 님의 음식 취향이 궁금해요', additional_kwargs={}, response_metadata={})]

ChatPromptTemplate을 이용하여 자신만의 프롬프트를 만들어 보세요.

In [14]:
my_chat_prompt = ChatPromptTemplate(
    [
        ("system", "당신은 한국 역사에 대해 전문적인 지식을 가진 전문 AI 어시스턴트입니다. 이름은 {name}입니다."),
        ("human", "안녕하세요 저는 {history_keyword}에 대해 궁금해요!"),
        ("ai", "같이 한 번 알아볼까요? 어떤게 궁금하신가요?"),
        ("human", "{question}")
    ]
)